## Jugando con los datos del case study based on Lambrechts (2008)

In [ ]:
!pip install dice-ml

In [ ]:
!pip install xgboost

In [60]:
## Libraries

# Sklearn imports
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV


# DiCE imports
import dice_ml
from dice_ml.utils import helpers  # helper functions

# pandas and numpy
import pandas as pd
import numpy as np

In [61]:
# Carga del dataset de simulaciones
data=pd.read_csv('./data/simulation_EV0.75_5-rand.csv',index_col=0)
data['critical_path']=data['critical_path'].astype('str')

# Expected time of the project 13
data['delay']=(data['duration']>13)*1

for i in range(1, 9):
    col_name = f't{i}_on_cp'
    data[col_name] = data['critical_path'].apply(lambda x: 1 if str(i) in x else 0)

In [62]:
data.head(2)

,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration@1,duration@2,...,critical_path,delay,t1_on_cp,t2_on_cp,t3_on_cp,t4_on_cp,t5_on_cp,t6_on_cp,t7_on_cp,t8_on_cp
0,1.890262,4.981053,6.176876,2.525289,4.874607,4.314132,4.488550,2.264787,1.890262,4.981053,...,368,0,0,0,1,0,0,1,0,1
1,1.923657,2.671438,5.377826,2.293507,17.989498,4.509809,8.463748,2.082815,1.923657,2.671438,...,259,1,0,1,0,0,1,0,0,0


#### Gradient boosting clasificator

In [63]:
# random seed
seed=112358

### RETRASO en función de estado de las tareas@ (y=D^BAC)
DBACdelay~ {activity i's duration at 75%EV} i=1,...,8

In [64]:
y=data.loc[:,'delay']
X=data.loc[:,['duration@1','duration@2', 'duration@3','duration@4', 'duration@5','duration@6', 'duration@7','duration@8']]

subsetdata=data.loc[:,['duration@1','duration@2', 'duration@3','duration@4', 'duration@5','duration@6', 'duration@7','duration@8','delay']]

In [65]:
# outer kfolds
kfold =StratifiedKFold(n_splits=5, random_state=seed,shuffle=True)

In [66]:
# results
results = pd.DataFrame([],columns=['model','kf','Accuracy'])

Para hacer model selection y tuneado de hiper parámetros haríamos como en sheva_model_seletion.

Voy a partir del gradient boosting y los parámetros que ya ajustamos:
 {'max_depth': 7, 'n_estimators': 100}

In [67]:
# Training 
mdC_dbac = GradientBoostingClassifier(max_depth=7, n_estimators=100,random_state=seed)
mdC_dbac.fit(X,y)

GradientBoostingClassifier(max_depth=7, random_state=112358)

In [68]:
# Training 
mdC_dbac = GradientBoostingClassifier(max_depth=7, n_estimators=100,random_state=seed)
mdC_dbac.fit(X,y)

GradientBoostingClassifier(max_depth=7, random_state=112358)

In [69]:
accuracy_score(y,mdC_dbac.predict(X))

0.89156

## DiCE can generate counterfactual examples using the following methods.

1) Model-agnostic methods

- Randomized sampling. 

- KD-Tree (for counterfactuals within the training data)

- Genetic algorithm


2) Gradient-based methods

- An explicit loss-based method described in Mothilal et al. (2020) (Default for deep learning models).

- A Variational AutoEncoder (VAE)-based method described in Mahajan et al. (2019) (see the BaseVAE notebook).

### 1.1 MODEL-AGNOSTIC, RANDOM

Muestrea dentro del rango de cada feature, no muestrea en el dataset original. No asegura feasibility.
https://github.com/interpretml/DiCE/blob/main/dice_ml/explainer_interfaces/dice_random.py

In [70]:
 #construct a data object for DiCE. 
d = dice_ml.Data(dataframe=subsetdata, continuous_features=['duration@1','duration@2', 'duration@3','duration@4', 'duration@5','duration@6', 'duration@7','duration@8'], outcome_name='delay')

# initiate DiCE
#We now initialize the DiCE explainer, which needs a dataset and a model. DiCE provides local explanation for the model m and requires an query input whose outcome needs to be explained.
# Using sklearn backend
m = dice_ml.Model(model=mdC_dbac, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml.Dice(d, m, method="random")

Vamos a generarle CF a la segunda instancia que va con retraso

In [71]:
subsetdata[1:2]

,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
1,1.923657,2.671438,5.377826,2.293507,5.580528,2.874139,4.034801,0.0,1


In [72]:
dice_exp = exp.generate_counterfactuals(X[1:2], total_CFs=8, desired_class="opposite")
dice_exp.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  4.86it/s]

Query instance (original outcome : 1)


,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
0,1.923657,2.671438,5.377826,2.293507,5.580528,2.874139,4.034801,0.0,1



Diverse Counterfactual set (new outcome: 0)


,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
0,-,-,-,-,-,0.4,-,-,0.0
1,-,-,-,-,-,3.3,1.0647531,-,0.0
2,0.8699234,-,-,-,-,-,1.3142302,-,0.0
3,-0.0474675,-,-,-,2.2043802,-,-,-,0.0
4,-,-,-,3.2442604,-,-,1.4090243,-,0.0
5,-,-,-,-,-,2.0,-,-,0.0
6,0.0296481,-,-,-,-,-,1.5225426,-,0.0
7,-,-,-,1.2669874,-,-,1.4962866,-,0.0


In [73]:
# get MAD
mads = d.get_mads(normalized=True)

# create feature weights
feature_weights = {}
for feature in mads:
    feature_weights[feature] = round(1/mads[feature], 2)
print(feature_weights)

{'duration@1': 13.73, 'duration@2': 12.88, 'duration@3': 11.37, 'duration@4': 12.97, 'duration@5': 13.63, 'duration@6': 10.19, 'duration@7': 11.75, 'duration@8': inf}


/var/folders/gm/dmfb8b7n71jgq4nczfq076s40000gn/T/ipykernel_43879/1464587245.py:7: RuntimeWarning: divide by zero encountered in divide
  feature_weights[feature] = round(1/mads[feature], 2)


He añadido dummy columns que nos digan si las tareas pertenecen al CP

### 1.2 MODEL-AGNOSTIC, KD-Tree (for counterfactuals within the training data)

https://github.com/interpretml/DiCE/blob/main/dice_ml/explainer_interfaces/dice_KD.py
Dice en la doc: "This code is similar to 'Interpretable Counterfactual Explanations Guided by Prototypes': https://arxiv.org/pdf/1907.02584.pdf"

El método con K-D tree busca ejemplos dentro del dataset que ya existen y pertenecen a la clase deseada.

In [75]:
exp_KD = dice_ml.Dice(d, m, method="kdtree")

dice_exp_KD = exp_KD.generate_counterfactuals(X[1:2], total_CFs=8, desired_class="opposite")
dice_exp_KD.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [01:33<00:00, 93.06s/it]

Query instance (original outcome : 1)


,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
0,1.923657,2.671438,5.377826,2.293507,5.580528,2.874139,4.034801,0.0,1



Diverse Counterfactual set (new outcome: 0)


,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
19326,2.1134038,2.7034814,5.5593448,2.1681583,5.4130526,2.6,3.8359721,-,0.0
905,2.1111042,2.8227432,5.4269799,2.2537101,5.2747431,2.7,3.7336721,-,0.0
39563,1.7459277,2.6318196,5.060709,2.1471803,5.1880851,2.8,3.9267965,-,0.0
48563,1.6598699,2.8275666,5.5390191,2.2233861,5.2241282,2.5,4.1694388,-,0.0
16671,1.6658391,2.7268075,5.5300689,2.0690262,5.298357,2.5,4.2912989,-,0.0
44120,1.8867921,2.5427682,5.817235,2.2508231,5.648354,2.4,4.0555071,-,0.0
34756,1.8522243,2.7594437,5.8303232,2.5952284,5.5703913,2.5,3.8823819,-,0.0
16246,1.5716493,2.4921889,5.5006818,2.6985207,5.3537531,2.6,3.8649848,-,0.0


### 2.1 GRADIENT METHODS. Explicit loss-based method described in Mothilal et al. (2020). Only for differenciable clasiffiers

No funciona con GradientBoostingClassifier: gradient is only supported for differentiable neural network models
Tampoco con XGBClassifier

In [ ]:
# Utilizo el gradient boosting que entrené antes
# https://github.com/interpretml/DiCE/blob/main/docs/source/notebooks/DiCE_getting_started.ipynb
#  
#construct a data object for DiCE. 
d = dice_ml.Data(dataframe=subsetdata, continuous_features=['duration@1','duration@2', 'duration@3','duration@4', 'duration@5','duration@6', 'duration@7','duration@8'], outcome_name='delay')

# initiate DiCE
#We now initialize the DiCE explainer, which needs a dataset and a model. DiCE provides local explanation for the model m and requires an query input whose outcome needs to be explained.
# Using sklearn backend
m = dice_ml.Model(model=mdC_dbac, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml.Dice(d, m, method="gradient")

UserConfigValidationException: gradient is only supported for differentiable neural network models. Please choose one of random, genetic or kdtree

In [ ]:
# Training XGBClassifier
from xgboost.sklearn import XGBClassifier
mdC_dbacXGBClassifier = XGBClassifier(max_depth=7, n_estimators=100,random_state=seed)
mdC_dbacXGBClassifier.fit(X,y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=112358, ...)

In [ ]:
m2 = dice_ml.Model(model=mdC_dbacXGBClassifier, backend="sklearn")
# Using method=random for generating CFs
exp = dice_ml.Dice(d, m2, method="gradient")

UserConfigValidationException: gradient is only supported for differentiable neural network models. Please choose one of random, genetic or kdtree

### Pruebo Tensorflow sin habermelo estudiado

In [ ]:
!pip install tensorflow

  Using cached tensorflow-2.18.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (4.0 kB)
  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.2.10-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-5.29.3-cp38-abi3-macosx_10_9_universal2.whl.metadata (592 bytes)
  Using cached termcolor-2.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached wrapt-1.17.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.4 kB)
  Using cached grpcio-1.70.0-cp312-cp312-macosx_10_14_universal2.whl.metadata (3.9 kB)
  Using cached tensorboard-2.18.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached keras-3.8.0

In [ ]:
# Training a tensorflow model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# define the keras model
tfmodel = Sequential()
tfmodel.add(Dense(12, input_dim=8, activation='relu'))
# compile the keras model
tfmodel.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy']) 
# fit tfmodel with fit(X,y)
tfmodel.add(Dense(8, activation='relu'))
tfmodel.add(Dense(1, activation='sigmoid'))
tfmodel.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
tfmodel.fit(X, y, epochs=10, batch_size=10, verbose=1)


Epoch 1/10


/opt/miniconda3/envs/my-env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 349us/step - accuracy: 0.7523 - loss: 0.5318
Epoch 2/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 333us/step - accuracy: 0.8094 - loss: 0.4069
Epoch 3/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 347us/step - accuracy: 0.8337 - loss: 0.3477
Epoch 4/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 339us/step - accuracy: 0.8452 - loss: 0.3211
Epoch 5/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 381us/step - accuracy: 0.8499 - loss: 0.3155
Epoch 6/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 342us/step - accuracy: 0.8478 - loss: 0.3178
Epoch 7/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 342us/step - accuracy: 0.8501 - loss: 0.3162
Epoch 8/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 347us/step - accuracy: 0.8545 - loss: 0.3111
Epoch 9/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 346us/step - accuracy: 0.8512 - loss: 0.3140
Epoch 10/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 2s 348us/step - accuracy: 0.8490 - loss: 0.3161


In [ ]:
# accuracy of the model
binary_predictions = (tfmodel.predict(X) > 0.5).astype(int)

# Compute the accuracy
accuracy = accuracy_score(y, binary_predictions)
print(f"Accuracy of the model: {accuracy}")

1563/1563 ━━━━━━━━━━━━━━━━━━━━ 0s 199us/step
Accuracy of the model: 0.85162


In [ ]:
backend = 'TF2'  # needs tensorflow installed
# dice_ml.Model
m = dice_ml.Model(model_path=tfmodel, backend=backend, func="ohe-min-max")

In [ ]:
dice_exp = exp.generate_counterfactuals(X[1:2], total_CFs=8, desired_class="opposite")
dice_exp.visualize_as_dataframe(show_only_changes=True)

100%|██████████| 1/1 [00:00<00:00,  4.32it/s]

Query instance (original outcome : 1)


,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
0,1.923657,2.671438,5.377826,2.293507,5.580528,2.874139,4.034801,0.0,1



Diverse Counterfactual set (new outcome: 0)


,duration@1,duration@2,duration@3,duration@4,duration@5,duration@6,duration@7,duration@8,delay
0,-,-,-,5.2498345,-,-,1.1944204,-,0.0
1,-,-,-,-,-,2.0,-,-,0.0
2,-,-,-,-0.083531,-,-,2.0134444,-,0.0
3,-,-,-,3.5574949,-,1.9,-,-,0.0
4,-,-,-,-,-,-,2.1397455,-,0.0
5,-,2.3347921,-,-,-,1.8,-,-,0.0
6,-,-,4.7453844,-,-,-,1.9312191,-,0.0
7,1.3131761,-,-,-,-,0.6,-,-,0.0
